# 04 — Regime Detection (Hidden Markov Model / Gaussian Mixture Model)
**GeoSentinel Terminal (VARTA) · Team 7 Lambda · SP2026**

Detects **Crisis vs Normal** market regimes using a Hidden Markov Model (HMM) or
Gaussian Mixture Model (GMM) trained on two features:
1. Brent crude 21-day rolling volatility
2. Global Database of Events Language and Tone (GDELT) news intensity score

Fallback: rolling volatility percentile threshold if Hidden Markov Model labels fail the economic sanity check.

Outputs: `data/processed/regimes.parquet`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [2]:
import numpy as np
import pandas as pd
import polars as pl
from hmmlearn import hmm
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from config import (
    DATA_PROC, REGIME_CRISIS, REGIME_NORMAL,
    HMM_N_COMPONENTS, HMM_N_ITER, HMM_COVARIANCE,
    GMM_N_COMPONENTS, GMM_MAX_ITER,
    GPR_SMOOTH_WINDOW,
)
from src.utils import log, save_parquet, set_dark_theme, annotate_events

log.info("Loading processed data...")

2026-04-16 08:47:47 [INFO] varta — Loading processed data...


In [3]:
# ── Load Brent prices and GDELT tone ──────────────────────────────────────────
fred  = pl.read_parquet(DATA_PROC / "fred.parquet")
gdelt = pl.read_parquet(DATA_PROC / "gdelt.parquet")

# Brent daily prices
brent = (
    fred.filter(pl.col("series_id") == "OIL_BRENT")
    .sort("date")
    .with_columns([
        pl.col("value").pct_change().alias("brent_return"),
    ])
    .with_columns([
        pl.col("brent_return").rolling_std(window_size=21).alias("brent_vol_21d"),
    ])
    .drop_nulls("brent_vol_21d")
)

# GDELT daily average tone (negative tone = more negative/crisis sentiment)
gdelt_daily = (
    gdelt
    .with_columns(pl.col("date").cast(pl.Date))
    .group_by("date")
    .agg([
        pl.col("tone").cast(pl.Float64).mean().alias("avg_tone"),
        pl.len().alias("n_articles"),
    ])
    .sort("date")
)

log.info(f"Brent vol series: {len(brent):,} rows")
log.info(f"GDELT daily tone: {len(gdelt_daily):,} days")

2026-04-16 08:47:47 [INFO] varta — Brent vol series: 3,629 rows


2026-04-16 08:47:47 [INFO] varta — GDELT daily tone: 60 days


In [4]:
# ── Build feature matrix ──────────────────────────────────────────────────────
# Join Brent vol + GDELT tone on date (left join — keep all Brent dates)
feature_df = (
    brent.select(["date", "value", "brent_vol_21d"])
    .rename({"value": "brent_price"})
    .with_columns(pl.col("date").cast(pl.Date))   # normalize to Date type
    .join(gdelt_daily, on="date", how="left")
    .with_columns([
        pl.col("avg_tone").forward_fill(limit=7).alias("avg_tone"),
        (-pl.col("avg_tone").forward_fill(limit=7)).alias("neg_tone"),
    ])
    .drop_nulls(["brent_vol_21d"])
    .fill_null(0.0)
)

print(f"Feature matrix: {feature_df.shape}")
feature_df.head()

Feature matrix: (3629, 6)


date,brent_price,brent_vol_21d,avg_tone,n_articles,neg_tone
date,f64,f64,f64,f64,f64
2010-08-31,75.51,0.020378,0.0,0.0,0.0
2010-09-01,75.53,0.01965,0.0,0.0,0.0
2010-09-02,74.93,0.019603,0.0,0.0,0.0
2010-09-03,75.03,0.019614,0.0,0.0,0.0
2010-09-07,75.78,0.019545,0.0,0.0,0.0


In [5]:
# ── Fit Hidden Markov Model ────────────────────────────────────────────────────
X = feature_df.select(["brent_vol_21d", "neg_tone"]).to_numpy()
X_scaled = StandardScaler().fit_transform(X)

model_hmm = hmm.GaussianHMM(
    n_components=HMM_N_COMPONENTS,
    covariance_type=HMM_COVARIANCE,
    n_iter=HMM_N_ITER,
    random_state=42,
)
model_hmm.fit(X_scaled)
hidden_states = model_hmm.predict(X_scaled)

# Map state → regime: state with HIGHER mean Brent vol = Crisis
state_vols = {s: X[hidden_states == s, 0].mean() for s in range(HMM_N_COMPONENTS)}
crisis_state = max(state_vols, key=state_vols.get)
normal_state  = min(state_vols, key=state_vols.get)
log.info(f"Hidden Markov Model crisis state: {crisis_state} (mean vol={state_vols[crisis_state]:.4f})")
log.info(f"Hidden Markov Model normal state: {normal_state} (mean vol={state_vols[normal_state]:.4f})")

hmm_labels = [REGIME_CRISIS if s == crisis_state else REGIME_NORMAL for s in hidden_states]
hmm_probs  = model_hmm.predict_proba(X_scaled)[:, crisis_state]

print(f"Crisis periods: {hmm_labels.count(REGIME_CRISIS)} days ({hmm_labels.count(REGIME_CRISIS)/len(hmm_labels)*100:.1f}%)")

2026-04-16 08:47:47 [INFO] varta — Hidden Markov Model crisis state: 1 (mean vol=0.0214)


2026-04-16 08:47:47 [INFO] varta — Hidden Markov Model normal state: 0 (mean vol=0.0206)


Crisis periods: 154 days (4.2%)


In [6]:
# ── SANITY CHECK — plot regime labels on Brent price chart ────────────────────
# Do crisis labels align with known shock dates (2014, 2020, 2022)?
# If YES → use Hidden Markov Model. If labels look wrong → use rolling vol fallback below.
import plotly.graph_objects as go

dates = feature_df["date"].to_pandas()
brent_prices = feature_df["brent_price"].to_numpy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=brent_prices, name="Brent Crude", line=dict(color="#A78BFA")))

# Shade crisis periods
crisis_mask = np.array(hmm_labels) == REGIME_CRISIS
for i in range(1, len(crisis_mask)):
    if crisis_mask[i] and not crisis_mask[i-1]:
        start = dates.iloc[i]
    if not crisis_mask[i] and crisis_mask[i-1]:
        end = dates.iloc[i]
        fig.add_vrect(x0=start, x1=end, fillcolor="#EF4444", opacity=0.15, line_width=0)

fig.update_layout(title="Hidden Markov Model Regime Labels on Brent Crude — Sanity Check", xaxis_title="Date", yaxis_title="Brent Price (USD)")
fig = set_dark_theme(fig)
fig = annotate_events(fig)
fig.show()
print("\n⚠️  Check: do red shaded regions align with 2014 oil crash, 2020 COVID, 2022 Ukraine?")
print("If YES → run next cell. If NO → skip to rolling vol fallback cell.")


⚠️  Check: do red shaded regions align with 2014 oil crash, 2020 COVID, 2022 Ukraine?
If YES → run next cell. If NO → skip to rolling vol fallback cell.


In [7]:
# ── USE THIS if Hidden Markov Model passes sanity check ──────────────────────
final_labels = hmm_labels
final_probs  = hmm_probs
method_used  = "HMM"
log.info(f"Using Hidden Markov Model labels — method: {method_used}")

2026-04-16 08:47:48 [INFO] varta — Using Hidden Markov Model labels — method: HMM


In [8]:
# ── FALLBACK — rolling volatility percentile threshold (run INSTEAD if HMM fails) ──
# Uncomment and run this cell if Hidden Markov Model labels fail the sanity check

# THRESHOLD_PCT = 75  # top 25% volatility days = Crisis regime
# vol_array = feature_df["brent_vol_21d"].to_numpy()
# threshold = np.percentile(vol_array, THRESHOLD_PCT)
# final_labels = [REGIME_CRISIS if v >= threshold else REGIME_NORMAL for v in vol_array]
# final_probs  = (vol_array - vol_array.min()) / (vol_array.max() - vol_array.min())
# method_used  = "rolling_vol_percentile"
# log.info(f"Using rolling vol fallback — threshold: {threshold:.4f} ({THRESHOLD_PCT}th percentile)")

In [9]:
# ── Build regime DataFrame and save ──────────────────────────────────────────
regimes_df = pl.DataFrame({
    "date":          feature_df["date"],
    "regime_label":  final_labels,
    "regime_prob":   final_probs,
    "method":        [method_used] * len(final_labels),
})

# Validate
assert set(regimes_df["regime_label"].unique().to_list()) == {REGIME_CRISIS, REGIME_NORMAL}, \
    "Both regime labels must appear at least once"
assert regimes_df["regime_prob"].min() >= 0 and regimes_df["regime_prob"].max() <= 1, \
    "Regime probabilities must be in [0, 1]"

crisis_pct = regimes_df.filter(pl.col("regime_label") == REGIME_CRISIS).shape[0] / len(regimes_df) * 100
log.info(f"Crisis: {crisis_pct:.1f}% | Normal: {100-crisis_pct:.1f}%")

save_parquet(regimes_df, DATA_PROC / "regimes.parquet", "regime labels")
print("Saved → data/processed/regimes.parquet")

2026-04-16 08:47:48 [INFO] varta — Crisis: 4.2% | Normal: 95.8%


2026-04-16 08:47:48 [INFO] varta — Saved regime labels → /Users/taruntheegela/Desktop/VARTA/data/processed/regimes.parquet (3,629 rows)


Saved → data/processed/regimes.parquet
